In [14]:

import pandas as pd

dataset = pd.read_excel('DUMMY_DATASET.xlsx')

# -------------------------
# 1. Build dataset
# -------------------------

subset = dataset[['YeshuvAvoda', 'ShemAvoda', 'SugAvoda', 'ShemMachlaka','EzoAvoda', 'MaamadAvoda', 'MakorSachar', 'TeudaGvoha', 'shnotlimud', 'Gil','SemelMishlachSofi']]

subset_dropped = subset.dropna()
subset_filtered = subset_dropped[~subset_dropped["SemelMishlachSofi"].str.contains(r"X", na=False)]

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor

# -------------------------
# 2. Split features/target
# -------------------------
X = subset_filtered.drop(columns=["SemelMishlachSofi"])
y = subset_filtered["SemelMishlachSofi"]

# -------------------------
# 3. Define column types
# -------------------------
cat_cols = ['YeshuvAvoda', 'ShemAvoda', 'SugAvoda', 'ShemMachlaka','EzoAvoda']
num_cols = ['MaamadAvoda', 'MakorSachar', 'TeudaGvoha', 'shnotlimud', 'Gil']

# -------------------------
# 4. Build preprocessing
# -------------------------

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

# -------------------------
# 5. Build model pipeline
# -------------------------


model = Pipeline([
    ("prep", preprocess),
    ("reg", RandomForestRegressor(
        n_estimators=100,
        max_depth=4,
        min_samples_leaf=3,
        random_state=42
    ))
])


# -------------------------
# 6. Train/test split
# -------------------------


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# -------------------------
# 7. Train model
# -------------------------
model.fit(X_train, y_train)

# -------------------------
# 8. Predict
# -------------------------
preds = model.predict(X_test)

print("Predictions:", preds)

# -------------------------
# 9. Evaluate
# -------------------------


mse = mean_squared_error(y_test, preds)
print("MSE:", mse)

# -------------------------
# 10. Cross-validation
# -------------------------


scores = cross_val_score(model, X, y, cv=5, scoring="neg_mean_squared_error")
print("CV MSE:", -scores.mean())

Predictions: [3693.91475461 2490.73772788 3050.50002819 3882.30559068 3031.92088103
 3811.28383395 5467.02334762 3882.30559068 3161.88291165 3811.28383395
 3908.78225735 5462.95916203 3369.58544986 3698.79719466 2500.52010162
 3154.68602178 8115.5458752  2789.78139583 2500.52010162 3496.9071231
 2448.08740345 2539.21432083 5486.40279207 3908.78225735 5771.38844188
 8561.87378316 2954.15838573 3161.88291165 5837.30701331 8157.78233768]
MSE: 1457022.1423864379
CV MSE: 956253.6659930584


In [19]:
from sklearn.model_selection import GridSearchCV

# -------------------------
# 10. Tune model parameters
# -------------------------
param_grid = {
    "reg__n_estimators": [50, 100, 200, 300],
    "reg__max_depth": [20, 100],
    "reg__min_samples_leaf": [1, 3, 5]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

grid.fit(X, y)

print("Best params:", grid.best_params_)
print("Best RMSE:", np.sqrt(-grid.best_score_))

Best params: {'reg__max_depth': 20, 'reg__min_samples_leaf': 1, 'reg__n_estimators': 300}
Best RMSE: 876.187933098212
